# Demo: classificador de alimentos de pratos brasileiros

Carrega o modelo público do Hugging Face Hub e abre uma interface para testar. Não precisa de token nem de GPU.

O modelo reconhece **recortes** de 20 alimentos: envie uma imagem em que um único alimento ocupa quase todo o quadro.

In [ ]:
!pip install -q transformers gradio

In [ ]:
import json
from pathlib import Path

import gradio as gr
import torch
from huggingface_hub import hf_hub_download, snapshot_download
from transformers import AutoImageProcessor, AutoModelForImageClassification

REPO = "aitalo/convnext-tiny-comida-brasileira"

model = AutoModelForImageClassification.from_pretrained(REPO).eval()
processor = AutoImageProcessor.from_pretrained(REPO)
LIMIAR = json.load(open(hf_hub_download(REPO, "limiar.json"), encoding="utf-8"))["limiar"]
exemplos = sorted(str(p) for p in Path(snapshot_download(REPO, allow_patterns="exemplos/*")).glob("exemplos/*"))
print(f"{len(model.config.id2label)} classes | limiar de rejeição: {LIMIAR:.3f} | {len(exemplos)} exemplos")

## Função de predição

Abaixo do limiar de confiança (calibrado no conjunto de validação), o modelo recusa responder. O limiar mede baixa confiança; ele não garante a detecção de imagens que não são comida.

In [ ]:
@torch.no_grad()
def predict(image):
    inputs = processor(image.convert("RGB"), return_tensors="pt")
    probs = model(**inputs).logits[0].softmax(-1)
    conf, ids = probs.topk(3)
    top3 = [{"classe": model.config.id2label[int(i)], "confianca": float(c)} for c, i in zip(conf, ids)]
    rejeitado = top3[0]["confianca"] < LIMIAR
    return {
        "classe": None if rejeitado else top3[0]["classe"],
        "confianca": top3[0]["confianca"],
        "top3": top3,
        "rejeitado": rejeitado,
    }


from PIL import Image

predict(Image.open(exemplos[0])) if exemplos else None

## Interface

`share=True` gera um link público temporário, que funciona enquanto esta célula estiver rodando.

In [ ]:
def interface(image):
    if image is None:
        return {}, "Envie uma imagem."
    out = predict(image)
    scores = {t["classe"]: t["confianca"] for t in out["top3"]}
    classe, conf = out["classe"], out["confianca"]
    if out["rejeitado"]:
        msg = f"**Rejeitado:** confiança {conf:.0%}, abaixo do limiar de {LIMIAR:.0%}. Pode não ser um dos 20 alimentos, ou não ser comida."
    else:
        msg = f"**{classe}** ({conf:.0%} de confiança)"
    return scores, msg


demo = gr.Interface(
    fn=interface,
    inputs=gr.Image(type="pil", label="Recorte de um alimento"),
    outputs=[gr.Label(num_top_classes=3, label="Top-3"), gr.Markdown()],
    examples=exemplos or None,
    title="Alimentos de pratos brasileiros",
    description="ConvNeXt-Tiny ajustado em 20 classes. Funciona melhor com um único alimento ocupando o quadro.",
    flagging_mode="never",
)
demo.launch(share=True)

## Créditos

As imagens de exemplo são recortes do dataset *Brazilian food images* (de Carvalho, Marcos Alberto, 2023, Mendeley Data, V2, doi: 10.17632/7n36jtcpv3.2), licença CC BY 4.0.